In [2]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install random

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement random (from versions: none)
ERROR: No matching distribution found for random


In [5]:
%pip install psycopg[binary]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import psycopg

In [7]:
from psycopg import sql

In [8]:
import random
import string
import pandas as pd


def generate_site_code():
    letters = "".join(random.choices(string.ascii_uppercase, k=3))
    numbers = "".join(random.choices(string.digits, k=3))
    return letters + numbers


def generate_site_data(count):
    data = []
    used_site_codes = set()
    used_coordinates = set()

    while len(data) < count:

        site_code = generate_site_code()
        latitude = round(random.uniform(-90, 90), 2)
        longitude = round(random.uniform(-180, 180), 2)

        # Skip duplicate site codes
        if site_code in used_site_codes:
            continue

        # Skip duplicate coordinates
        if (latitude, longitude) in used_coordinates:
            continue

        used_site_codes.add(site_code)
        used_coordinates.add((latitude, longitude))

        data.append(
            {
                "site_code": site_code,
                "latitude": latitude,
                "longitude": longitude,
            }
        )

    return pd.DataFrame(data)

In [9]:
def create_db_meta(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [10]:
create_db_meta("meta")

Database 'meta' created successfully!


In [11]:
def create_table_meta():
    try:
        with psycopg.connect(
            dbname="meta",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS metadata (
                        site_name VARCHAR(100) Not NULL,
                        latitude DOUBLE PRECISION Not NULL,
                        longitude DOUBLE PRECISION Not NULL
                    );
                """)

            conn.commit()
            print("metadata table created.")

    except psycopg.Error as e:
        print(e)

In [12]:
create_table_meta()

metadata table created.


In [13]:
def insert_sites():

    sites_df = generate_site_data(10000)

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:
        with conn.cursor() as cur:
            for _, row in sites_df.iterrows():
                cur.execute(
                    """
                    INSERT INTO metadata
                    (site_name, latitude, longitude)
                    VALUES (%s, %s, %s)                                                                         
                    """,
                    (row["site_code"], row["latitude"], row["longitude"]),
                )
        conn.commit()

    print("Sites inserted successfully!")

In [14]:
insert_sites()

Sites inserted successfully!


In [15]:
def get_sites():

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute(""" 
                SELECT site_name, latitude, longitude
                FROM metadata
            """)

            sites = cur.fetchall()

    return sites

In [16]:
data = get_sites()

In [17]:
type(data[0])

tuple

In [18]:
len(data)

10000

In [19]:
def create_db_site_weather(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")
        

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [20]:
create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [21]:
def create_table_site_weather():
    try:
        with psycopg.connect(
            dbname="site_weather",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS site_weather (
                        site_name VARCHAR(100) NOT NULL,
                        time_interval TIMESTAMPTZ NOT NULL,
                        temperature REAL NOT NULL,
                        humidity REAL NOT NULL,
                        solar_radiance REAL NOT NULL,

                        UNIQUE (site_name, time_interval)
                    );
                """)

        print("site_weather created successfully.")

    except psycopg.Error as e:
        print(e)

In [22]:
create_table_site_weather()

site_weather created successfully.


In [20]:
import time
import datetime
import requests
import psycopg

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

DB_NAME = "site_weather"

API_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

BATCH_SIZE = 30

REQUEST_DELAY = 0.5

RETRY_WAIT = 5

RATE_LIMIT_WAIT = 60

REQUEST_TIMEOUT = 120

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS site_weather (
                site_name VARCHAR(100) NOT NULL,
                time_interval TIMESTAMPTZ NOT NULL,
                temperature REAL NOT NULL,
                humidity REAL NOT NULL,
                solar_radiance REAL NOT NULL,

                UNIQUE (site_name, time_interval)
            );
        """)

    conn.commit()


with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT site_name
            FROM site_weather
            GROUP BY site_name
            HAVING COUNT(*) = 24;
        """)

        completed_sites = {row[0] for row in cur.fetchall()}

pending_sites = []

for row in data:

    site_name = row[0]
    latitude = float(row[1])
    longitude = float(row[2])

    if site_name not in completed_sites:
        pending_sites.append((site_name, latitude, longitude))


# print("=" * 60)
# print("RESUME CHECK")
print("=" * 60)
print(f"Total sites       : {len(data)}")
print(f"Completed sites   : {len(completed_sites)}")
print(f"Remaining sites   : {len(pending_sites)}")
print("=" * 60)


def fetch_batch(batch):

    site_names = [site[0] for site in batch]
    latitudes = [str(site[1]) for site in batch]
    longitudes = [str(site[2]) for site in batch]

    params = {
        "latitude": ",".join(latitudes),
        "longitude": ",".join(longitudes),
        "hourly": ",".join(
            [
                "temperature_2m",
                "relative_humidity_2m",
                "direct_radiation",
            ]
        ),
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    # Keep trying the SAME batch until it succeeds.
    while True:

        try:

            # print()
            # print(f"Requesting {len(batch)} sites...")
            # print(f"{site_names[0]} -> {site_names[-1]}")

            response = requests.get(
                API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code == 200:

                result = response.json()

                if isinstance(result, dict):
                    result = [result]

                if not isinstance(result, list):
                    raise ValueError("Invalid API response format.")

                if len(result) != len(batch):
                    raise ValueError(
                        f"Expected {len(batch)} locations, "
                        f"but API returned {len(result)}."
                    )

                print("API SUCCESS")
                return result

            if response.status_code == 429:

                print(
                    f"API rate limit reached. " f"Waiting {RATE_LIMIT_WAIT} seconds..."
                )

                time.sleep(RATE_LIMIT_WAIT)

                print("Retrying SAME batch...")
                continue

            if response.status_code >= 500:

                print(
                    f"Server error {response.status_code}. "
                    f"Waiting {RETRY_WAIT} seconds..."
                )

                time.sleep(RETRY_WAIT)

                print("Retrying SAME batch...")
                continue

            print(
                f"API error {response.status_code}. " f"Waiting {RETRY_WAIT} seconds..."
            )

            time.sleep(RETRY_WAIT)

            print("Retrying SAME batch...")

        except (requests.Timeout, requests.ConnectionError) as e:

            print(f"Network error: {e}")
            print(f"Waiting {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)
            print("Retrying SAME batch...")

        except (requests.RequestException, ValueError) as e:

            print(f"Request/response error: {e}")
            print(f"Waiting {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)
            print("Retrying SAME batch...")


def extract_site_weather(site_name, weather):

    hourly = weather.get("hourly")

    if not hourly:
        raise ValueError(f"No hourly data for {site_name}")

    times = hourly.get("time", [])
    temperatures = hourly.get("temperature_2m", [])
    humidity = hourly.get("relative_humidity_2m", [])
    radiation = hourly.get("direct_radiation", [])

    # We need exactly 24 records for one day.
    if not (
        len(times) == 24
        and len(temperatures) == 24
        and len(humidity) == 24
        and len(radiation) == 24
        
    ):
        raise ValueError(
            f"Invalid data for {site_name}: "
            f"time={len(times)}, "
            f"temperature={len(temperatures)}, "
            f"humidity={len(humidity)}, "
            f"radiation={len(radiation)}"
        )

    rows = []

    for i in range(24):

        timestamp = datetime.datetime.fromisoformat(times[i]).replace(
            tzinfo=datetime.timezone.utc
        )

        rows.append(
            (
                site_name,
                timestamp,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            )
        )

    return rows


def save_batch(cursor, batch, api_results):

    all_rows = []

    for i, site in enumerate(batch):

        site_name = site[0]
        weather = api_results[i]

        rows = extract_site_weather(
            site_name,
            weather,
        )

        all_rows.extend(rows)

    cursor.executemany(
        """
        INSERT INTO site_weather (
            site_name,
            time_interval,
            temperature,
            humidity,
            solar_radiance
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (site_name, time_interval)
        DO NOTHING;
    """,
        all_rows,
    )

    return len(all_rows)


total_rows = 0
total_pending = len(pending_sites)

if total_pending == 0:

    print("All sites are already complete.")

else:

    total_batches = (total_pending + BATCH_SIZE - 1) // BATCH_SIZE

    with psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cur:

            for batch_number, start in enumerate(
                range(0, total_pending, BATCH_SIZE),
                start=1,
            ):

                batch = pending_sites[start : start + BATCH_SIZE]

                # print()
                # print("=" * 60)
                # print(f"BATCH {batch_number}/{total_batches}")
                # print(f"Sites: {start + 1} - " f"{start + len(batch)}")
                # print("=" * 60)

                # API

                api_results = fetch_batch(batch)

                # DATABASE

                rows_inserted = save_batch(
                    cur,
                    batch,
                    api_results,
                )

                conn.commit()

                total_rows += rows_inserted

                print(
                    f"Batch complete | "
                    f"Sites: {len(batch)} | "
                    f"Rows: {rows_inserted} | "
                    f"Total rows: {total_rows}"
                )

                time.sleep(REQUEST_DELAY)

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT COUNT(DISTINCT site_name) 
            FROM site_weather;
        """)

        actual_sites = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM site_weather;   
        """)

        actual_rows = cur.fetchone()[0]


expected_sites = len(data)
expected_rows = expected_sites * 24

print()
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)
print(f"Expected sites : {expected_sites}")
print(f"Actual sites   : {actual_sites}")
print(f"Expected rows  : {expected_rows}")
print(f"Actual rows    : {actual_rows}")
print("=" * 60)

if actual_sites == expected_sites and actual_rows == expected_rows:
    print("SUCCESS: All sites and all hourly records are complete.")
else:
    print("NOT COMPLETE YET.")
    print(f"Missing rows: {expected_rows - actual_rows}")

Total sites       : 10000
Completed sites   : 0
Remaining sites   : 10000
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 720
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 1440
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 2160
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 2880
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 3600
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 4320
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 5040
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 5760
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 6480
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 7200
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 7920
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 8640
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 9360
API SUCCESS
Batch

In [ ]:
import time
import datetime
import requests
import psycopg

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

DB_NAME = "site_weather"

API_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

BATCH_SIZE = 30

REQUEST_DELAY = 0.5

RETRY_WAIT = 5

RATE_LIMIT_WAIT = 60

REQUEST_TIMEOUT = 120

start_datetime = datetime.datetime.fromisoformat(START_DATE).replace(
    tzinfo=datetime.timezone.utc
)

end_datetime = (
    datetime.datetime.fromisoformat(END_DATE) + datetime.timedelta(days=1)
).replace(tzinfo=datetime.timezone.utc)

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute(
            """
            SELECT site_name
            FROM site_weather
            WHERE time_interval >= %s
              AND time_interval < %s
            GROUP BY site_name
            HAVING COUNT(*) = 24;
            """,
            (
                start_datetime,
                end_datetime,
            ),
        )

        completed_sites = {row[0] for row in cur.fetchall()}


# ============================================================
# BUILD PENDING SITE LIST
# ============================================================

pending_sites = []

for row in data:

    site_name = row[0]
    latitude = float(row[1])
    longitude = float(row[2])

    if site_name not in completed_sites:
        pending_sites.append(
            (
                site_name,
                latitude,
                longitude,
            )
        )

print("=" * 60)
print("RESUME CHECK")
print("=" * 60)
print(f"Total sites       : {len(data)}")
print(f"Completed sites   : {len(completed_sites)}")
print(f"Remaining sites   : {len(pending_sites)}")
print("=" * 60)


# ============================================================
# CHECK ONE SITE IN DATABASE
#
# This is the important resume/checkpoint function.
# ============================================================


def site_has_24_records(cursor, site_name):

    cursor.execute(
        """
        SELECT COUNT(*)
        FROM site_weather
        WHERE site_name = %s
          AND time_interval >= %s
          AND time_interval < %s;
        """,
        (
            site_name,
            start_datetime,
            end_datetime,
        ),
    )

    count = cursor.fetchone()[0]

    return count == 24


# ============================================================
# FETCH BATCH
# ============================================================


def fetch_batch(batch):

    site_names = [site[0] for site in batch]

    latitudes = [str(site[1]) for site in batch]

    longitudes = [str(site[2]) for site in batch]

    params = {
        "latitude": ",".join(latitudes),
        "longitude": ",".join(longitudes),
        "hourly": ",".join(
            [
                "temperature_2m",
                "relative_humidity_2m",
                "direct_radiation",
            ]
        ),
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    # Keep retrying the SAME batch until the HTTP
    # request itself succeeds.

    while True:

        try:

            response = requests.get(
                API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code == 200:

                result = response.json()

                if isinstance(result, dict):
                    result = [result]

                if not isinstance(result, list):
                    raise ValueError("Invalid API response format.")

                if len(result) != len(batch):
                    raise ValueError(
                        f"Expected {len(batch)} locations, "
                        f"but API returned {len(result)}."
                    )

                print(f"API SUCCESS | " f"Batch sites: {len(batch)}")

                return result

            # ------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------

            if response.status_code == 429:

                print(
                    "API rate limit reached. " f"Waiting {RATE_LIMIT_WAIT} seconds..."
                )

                time.sleep(RATE_LIMIT_WAIT)

                continue

            # ------------------------------------------------
            # SERVER ERROR
            # ------------------------------------------------

            if response.status_code >= 500:

                print(
                    f"Server error {response.status_code}. "
                    f"Waiting {RETRY_WAIT} seconds..."
                )

                time.sleep(RETRY_WAIT)

                continue

            # ------------------------------------------------
            # OTHER HTTP ERROR
            # ------------------------------------------------

            print(
                f"API error {response.status_code}. " f"Waiting {RETRY_WAIT} seconds..."
            )

            time.sleep(RETRY_WAIT)

        except (
            requests.Timeout,
            requests.ConnectionError,
        ) as e:

            print(f"Network error: {e}")
            print(f"Waiting {RETRY_WAIT} seconds...")

            time.sleep(RETRY_WAIT)

        except (
            requests.RequestException,
            ValueError,
        ) as e:

            print(f"Request/response error: {e}")

            print(f"Waiting {RETRY_WAIT} seconds...")

            time.sleep(RETRY_WAIT)


# ============================================================
# EXTRACT ONE SITE
# ============================================================


def extract_site_weather(
    site_name,
    weather,
):

    # API can return an error object for an individual
    # location.

    if weather.get("error"):

        reason = weather.get("reason", "Unknown API error")

        raise ValueError(f"API error for {site_name}: {reason}")

    hourly = weather.get("hourly")

    if not hourly:

        raise ValueError(f"No hourly data for {site_name}")

    times = hourly.get("time", [])

    temperatures = hourly.get("temperature_2m", [])

    humidity = hourly.get("relative_humidity_2m", [])

    radiation = hourly.get("direct_radiation", [])

    # --------------------------------------------------------
    # EACH SITE MUST HAVE EXACTLY 24 RECORDS
    # --------------------------------------------------------

    if not (
        len(times) == 24
        and len(temperatures) == 24
        and len(humidity) == 24
        and len(radiation) == 24
    ):

        raise ValueError(
            f"Invalid data for {site_name}: "
            f"time={len(times)}, "
            f"temperature={len(temperatures)}, "
            f"humidity={len(humidity)}, "
            f"radiation={len(radiation)}"
        )

    rows = []

    for i in range(24):

        timestamp = datetime.datetime.fromisoformat(times[i]).replace(
            tzinfo=datetime.timezone.utc
        )

        rows.append(
            (
                site_name,
                timestamp,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            )
        )

    return rows


# ============================================================
# SAVE ONE SITE
#
# We commit each site separately.
#
# THIS IS IMPORTANT FOR RESUME.
#
# If the program crashes after site 200,
# sites 1-200 are already committed.
# ============================================================


def save_site(
    cursor,
    site_name,
    weather,
):

    rows = extract_site_weather(
        site_name,
        weather,
    )

    cursor.executemany(
        """
        INSERT INTO site_weather (
            site_name,
            time_interval,
            temperature,
            humidity,
            solar_radiance
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (
            site_name,
            time_interval
        )
        DO NOTHING;
        """,
        rows,
    )

    return len(rows)


# ============================================================
# FETCH A SINGLE SITE
#
# Used when an individual site inside a batch fails.
#
# This means one bad site does NOT cause the whole batch
# to be considered failed.
# ============================================================


def fetch_single_site(site):

    while True:

        try:

            result = fetch_batch([site])

            weather = result[0]

            # Validate the individual site immediately.

            extract_site_weather(
                site[0],
                weather,
            )

            return weather

        except ValueError as e:

            print(f"Individual site validation failed " f"for {site[0]}: {e}")

            print(f"Waiting {RETRY_WAIT} seconds...")

            time.sleep(RETRY_WAIT)

        except Exception as e:

            print(f"Error fetching {site[0]}: {e}")

            print(f"Waiting {RETRY_WAIT} seconds...")

            time.sleep(RETRY_WAIT)


# ============================================================
# MAIN PROCESSING
# ============================================================

total_rows = 0

total_pending = len(pending_sites)


if total_pending == 0:

    print("All sites are already complete.")

else:

    total_batches = (total_pending + BATCH_SIZE - 1) // BATCH_SIZE

    with psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cur:

            for batch_number, start in enumerate(
                range(
                    0,
                    total_pending,
                    BATCH_SIZE,
                ),
                start=1,
            ):

                batch = pending_sites[start : start + BATCH_SIZE]

                print()
                print("=" * 60)
                print(f"BATCH " f"{batch_number}/{total_batches}")
                print(f"Sites: " f"{start + 1} - " f"{start + len(batch)}")
                print("=" * 60)

                # ------------------------------------------------
                # FETCH BATCH
                # ------------------------------------------------

                api_results = fetch_batch(batch)

                # ------------------------------------------------
                # PROCESS EACH SITE INDIVIDUALLY
                # ------------------------------------------------

                for i, site in enumerate(batch):

                    site_name = site[0]

                    print(f"\nChecking site: " f"{site_name}")

                    # --------------------------------------------
                    # CHECK DATABASE AGAIN
                    #
                    # This protects against:
                    # - previous partial runs
                    # - partial batches
                    # - crashes
                    # --------------------------------------------

                    if site_has_24_records(
                        cur,
                        site_name,
                    ):

                        print(
                            f"SKIP | " f"{site_name} already has " f"24 hourly records."
                        )

                        continue

                    # --------------------------------------------
                    # TRY DATA FROM BATCH
                    # --------------------------------------------

                    weather = api_results[i]

                    try:

                        rows_inserted = save_site(
                            cur,
                            site_name,
                            weather,
                        )

                        # ----------------------------------------
                        # VERIFY DATABASE AFTER INSERT
                        # ----------------------------------------

                        if not site_has_24_records(
                            cur,
                            site_name,
                        ):

                            # Roll back this site's insert.

                            conn.rollback()

                            print(
                                f"WARNING | "
                                f"{site_name} does not have "
                                f"24 records after batch response."
                            )

                            print(f"Retrying {site_name} individually...")

                            # Fetch the site separately.

                            weather = fetch_single_site(site)

                            rows_inserted = save_site(
                                cur,
                                site_name,
                                weather,
                            )

                        # ----------------------------------------
                        # FINAL SITE VERIFICATION
                        # ----------------------------------------

                        if not site_has_24_records(
                            cur,
                            site_name,
                        ):

                            conn.rollback()

                            raise ValueError(
                                f"{site_name} still does not "
                                f"have exactly 24 records."
                            )

                        # ----------------------------------------
                        # COMMIT IMMEDIATELY
                        #
                        # THIS IS THE CHECKPOINT.
                        # ----------------------------------------

                        conn.commit()

                        total_rows += rows_inserted

                        completed_sites.add(site_name)

                        print(
                            f"SUCCESS | "
                            f"{site_name} | "
                            f"24/24 hours | "
                            f"Committed to database"
                        )

                    except Exception as e:

                        conn.rollback()

                        print(f"FAILED | " f"{site_name} | " f"{e}")

                        print(f"Fetching {site_name} " f"individually...")

                        # ----------------------------------------
                        # INDIVIDUAL RETRY
                        # ----------------------------------------

                        weather = fetch_single_site(site)

                        rows_inserted = save_site(
                            cur,
                            site_name,
                            weather,
                        )

                        # ----------------------------------------
                        # VERIFY AGAIN
                        # ----------------------------------------

                        if not site_has_24_records(
                            cur,
                            site_name,
                        ):

                            conn.rollback()

                            raise ValueError(
                                f"Could not obtain "
                                f"24 complete records "
                                f"for {site_name}"
                            )

                        # ----------------------------------------
                        # COMMIT THIS SITE
                        # ----------------------------------------

                        conn.commit()

                        total_rows += rows_inserted

                        completed_sites.add(site_name)

                        print(
                            f"SUCCESS | "
                            f"{site_name} | "
                            f"24/24 hours | "
                            f"Committed to database"
                        )

                    time.sleep(REQUEST_DELAY)


# ============================================================
# FINAL VERIFICATION
# ============================================================

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        # --------------------------------------------------------
        # Count completed sites for the requested date
        # --------------------------------------------------------

        cur.execute(
            """
            SELECT COUNT(*)
            FROM (
                SELECT site_name
                FROM site_weather
                WHERE time_interval >= %s
                  AND time_interval < %s
                GROUP BY site_name
                HAVING COUNT(*) = 24
            ) AS completed;
            """,
            (
                start_datetime,
                end_datetime,
            ),
        )

        actual_sites = cur.fetchone()[0]

        # --------------------------------------------------------
        # Count all rows for requested date
        # --------------------------------------------------------

        cur.execute(
            """
            SELECT COUNT(*)
            FROM site_weather
            WHERE time_interval >= %s
              AND time_interval < %s;
            """,
            (
                start_datetime,
                end_datetime,
            ),
        )

        actual_rows = cur.fetchone()[0]


expected_sites = len(data)

expected_rows = expected_sites * 24


print()
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print(f"Expected sites : " f"{expected_sites}")

print(f"Actual sites   : " f"{actual_sites}")

print(f"Expected rows  : " f"{expected_rows}")

print(f"Actual rows    : " f"{actual_rows}")

print("=" * 60)


if actual_sites == expected_sites and actual_rows == expected_rows:

    print("SUCCESS: All sites and " "all hourly records are complete.")

else:

    print("NOT COMPLETE YET.")

    print(f"Missing rows: " f"{expected_rows - actual_rows}")

RESUME CHECK
Total sites       : 10000
Completed sites   : 0
Remaining sites   : 10000

BATCH 1/334
Sites: 1 - 30
API SUCCESS | Batch sites: 30

Checking site: KYR754
SUCCESS | KYR754 | 24/24 hours | Committed to database

Checking site: FZS492
SUCCESS | FZS492 | 24/24 hours | Committed to database

Checking site: YPO895
SUCCESS | YPO895 | 24/24 hours | Committed to database

Checking site: NZY952
SUCCESS | NZY952 | 24/24 hours | Committed to database

Checking site: SKX985
SUCCESS | SKX985 | 24/24 hours | Committed to database

Checking site: OAY537
SUCCESS | OAY537 | 24/24 hours | Committed to database

Checking site: QDS775
SUCCESS | QDS775 | 24/24 hours | Committed to database

Checking site: STD032
SUCCESS | STD032 | 24/24 hours | Committed to database

Checking site: AZZ584
SUCCESS | AZZ584 | 24/24 hours | Committed to database

Checking site: AMB904
SUCCESS | AMB904 | 24/24 hours | Committed to database

Checking site: CGO428
SUCCESS | CGO428 | 24/24 hours | Committed to databas

In [7]:
# name = "elaf"; print(f"Hello {name}"); name = "muneeb"; print(f"{name}")

Hello elaf
muneeb


In [12]:
# string = "123"
# print(string.isalpha())
# print(string.isdecimal())
# print(string.isalnum())

False
True
True


In [ ]:
# import random

# def rock_paper_scissors():
#     choices = ["rock", "paper", "scissors"]
#     player_choice = input("Enter your choice(\"rock, paper, scissors\"): ")
#     computer_choice = random.choice(choices) 
    
#     result = {"player_choice": player_choice, "computer_choice": computer_choice}
#     print(result)
#     return result

# def check_win():
#     result = rock_paper_scissors()
    
#     if result["player_choice"] == result["computer_choice"]:
#         print("It is a tie!")
    
#     elif result["player_choice"] == "rock":
#         if result["computer_choice"] == "paper":
#             print("Computer choice paper which can wrap rock, computer wins!")
#         else:
#             print("Computer choose scissors which can be break by rock, player wins!")
            
#     elif result["player_choice"] == "paper":
#             if result["computer_choice"] == "scissors":
#                 print("Computer choice scissors which can cut paper, computer wins!")
#             else:
#                 print("Computer choose rock which can be wrapped by paper, player wins!")
    
#     elif result["player_choice"] == "scissors":
#             if result["computer_choice"] == "rock":
#                 print("Computer choice rock which can break scissors, computer wins!")
#             else:
#                 print("Computer choose paper which can be cut by scissors, player wins!")
    
    
# check_win()
    


{'player_choice': 'rock', 'computer_choice': 'paper'}
Computer choice paper which can wrap rock, computer wins!


In [ ]:
# string = "abcdef"
# print(string[1:])

bcdef
